In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bnet-dataset/pre_parsed_dataset.csv


# Blood Donation Message Classification

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import torch

In [3]:
df = pd.read_csv('/kaggle/input/bnet-dataset/pre_parsed_dataset.csv')

In [4]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Using fasttext

In [5]:
import fasttext

In [6]:

# Write the training data in FastText format (label + text)
with open("train.txt", "w") as f:
    for text, label in zip(X_train, y_train):
        # FastText expects labels to be prefixed with '__label__'
        f.write(f"__label__{label} {text}\n")

# Write the testing data in FastText format (for evaluation purposes)
with open("test.txt", "w") as f:
    for text, label in zip(X_test, y_test):
        f.write(f"__label__{label} {text}\n")

# trainining the fasttext classifier

In [7]:
# Train the FastText classifier
model = fasttext.train_supervised(
    input="train.txt",
    epoch=1000,              # Increase epochs for better learning
    lr=1.0,                # Adjust learning rate
    wordNgrams=3,           # Use trigrams for better context
    minn=3,                 # Minimum subword length
    maxn=6,                 # Maximum subword length
    verbose=2
)


In [8]:
# Save the model checkpoint to a file
model.save_model("bnet_classier_fasttext_model.bin")  # Saves the model to a binary file

In [9]:
# Evaluate the model using the test set
def evaluate_model(test_file):
    # FastText's built-in test function returns (n, precision, recall)
    result = model.test(test_file)
    n_samples = result[0]  # Number of samples
    precision = result[1]  # Precision
    recall = result[2]     # Recall
    
    # Calculate F1-score: 2 * (precision * recall) / (precision + recall)
    if precision + recall > 0:  # Avoid division by zero
        f1_score = 2 * (precision * recall) / (precision + recall)
    else:
        f1_score = 0.0
    
    # For binary classification, accuracy might align with precision in some cases,
    # but typically you need true positives/negatives for exact accuracy (see Option 2)
    print(f"Number of samples: {n_samples}")
    print(f"Precision: {precision:.10f}")
    print(f"Recall: {recall:.10f}")
    print(f"F1-Score: {f1_score:.10f}")
    # Note: Accuracy isn't directly provided; see Option 2 for a precise calculation

# Call the evaluation function
evaluate_model("test.txt")

Number of samples: 5181
Precision: 0.9739432542
Recall: 0.9739432542
F1-Score: 0.9739432542


In [10]:
# Function to make predictions on custom messages
def predict_messages(messages):
    for message in messages:
        prediction = model.predict(message)  # Predict the label
        label = prediction[0][0]  # Extract predicted label
        confidence = prediction[1][0]  # Extract confidence score
        print(f"Message: '{message}' | Prediction: {label} | Confidence: {confidence}")

# Testing with custom messages

In [11]:

# Test the model with your custom messages
messages = [
"হ্যা লো বন্ধুরা, এখন আমরা মডেলটিকে টেস্ট করব। চলো দেখি",
"ektu test kore dekhi kemon lage"
]

print(f"{predict_messages(messages)}")

Message: 'হ্যা লো বন্ধুরা, এখন আমরা মডেলটিকে টেস্ট করব। চলো দেখি' | Prediction: __label__0 | Confidence: 0.9988477826118469
Message: 'ektu test kore dekhi kemon lage' | Prediction: __label__0 | Confidence: 1.0000100135803223
None
